# Creating Train and Test Sets with Explicit Feedback
- Dataset: MovieLens32m
- Split Method: leave-one-out + 100 random samples


source: https://github.com/yihong-chen/neural-collaborative-filtering/blob/master/src/data.py

In [50]:
import numpy as np
import pandas as pd
import scipy.sparse as sp

np.random.seed(42)

In [51]:
def set_dtypes(dataset: pd.DataFrame) -> pd.DataFrame:
    dataset['userId'] = dataset['userId'].astype(np.uint32)
    dataset['rating'] = dataset['rating'].astype(np.float16)
    dataset['movieId'] = dataset['movieId'].astype(np.uint32)
    dataset['timestamp'] = dataset['timestamp'].astype(np.uint32)

    return dataset

In [52]:
ratings = pd.read_csv('../data/ml-32m/ratings.csv')
ratings = set_dtypes(ratings)

In [53]:
user_pool = set(ratings['userId'].unique())
item_pool = set(ratings['movieId'].unique())

### Remap IDs to be contiguous

In [54]:
user_id_map = {uid: idx for idx, uid in enumerate(ratings['userId'].unique())}
item_id_map = {mid: idx for idx, mid in enumerate(ratings['movieId'].unique())}

ratings['userId'] = ratings['userId'].map(user_id_map).astype(np.int64)
ratings['movieId'] = ratings['movieId'].map(item_id_map).astype(np.int64)

### Pop last rated item

In [55]:
def pop_last_user_review(ratings_df):
    # Sort by timestamp to ensure the last review is at the end
    ratings_df = ratings_df.sort_values(by='timestamp')
    
    # Get the last review for each user
    last_reviews = ratings_df.groupby('userId').tail(1)
    
    # Remove the last reviews from the original DataFrame
    ratings_without_last = ratings_df[~ratings_df.index.isin(last_reviews.index)]
    
    return ratings_without_last, last_reviews

In [56]:
ratings_without_last, last_ratings = pop_last_user_review(ratings)

### Export as train and test sets

In [57]:
ratings_without_last = ratings_without_last.drop(columns=['timestamp'])

In [58]:
last_ratings = last_ratings.drop(columns=['timestamp'])

In [59]:
ratings_without_last.to_csv("../data/train.csv", index=False)
last_ratings.to_csv("../data/test.csv", index=False)

# Below is not needed for explicit feedback

### Create DoK and Convert to CSR
- To store more efficiently and to be able to create training and test sets

In [ ]:
user_item_matrix = sp.dok_matrix((ratings['userId'].max() + 1, ratings['movieId'].max() + 1), dtype=np.float32)

In [ ]:
for user, item, rating, _ in ratings.itertuples(index=False):
    user_item_matrix[user, item] = rating

In [27]:
user_item_matrix = user_item_matrix.tocsr()

### Find all unrated items + sample 99

Below is from source but cannot be performed with this dataset due to memory constraints

In [ ]:
# def sample_negative(ratings):
#     """return all negative items & 99 sampled negative items"""
#     # Get all movies that a user has rated, for every user
#     interact_status = ratings.groupby('userId')['movieId'].apply(set).reset_index().rename(
#         columns={'movieId': 'interacted_items'})
#     # Get all movies that a user hasn't rated, for every user
#     interact_status['negative_items'] = interact_status['interacted_items'].apply(lambda x: item_pool - x)
#     # Get 99 random samples from all movies that a user hasn't interacted with
#     interact_status['negative_samples'] = interact_status['negative_items'].apply(lambda x: np.random.sample(list(x), 99))
#     return interact_status[['userId', 'negative_items', 'negative_samples']]

In [ ]:
# negatives = sample_negative(ratings)

Instead, do this using `scipy.sparse.csr_matrix`

In [ ]:
item_pool_list = list(item_pool)
negative_samples = {}

for user in user_pool:
    # Get the set of items the user has already rated
    # In CSR, indices for a row 'u' are stored between indptr[u] and indptr[u+1]
    rated_items = set(user_item_matrix.indices[user_item_matrix.indptr[user]:user_item_matrix.indptr[user+1]])
    
    user_negatives = set()
    
    # Randomly sample until we find 99 unrated movies
    while len(user_negatives) < 99:
        candidate = np.random.choice(item_pool_list)
        if candidate not in rated_items:
            user_negatives.add(candidate)
            
    negative_samples[user] = list(user_negatives)